# M2 練習 Notebook — Data Ingestion

本 notebook 分兩部分：
- **Part A（Mentor 示範）**：多表 join 的正確性驗證、pd.concat、函式介面設計
- **Part B（你來練習）**：空格讓你舉一反三

M1 已經練過 read_csv / merge 的基本用法，M2 聚焦在：
1. Join 之後怎麼驗證結果是正確的？
2. 多張同結構的表怎麼垂直合併？
3. 怎麼把這些操作包成一個有明確介面的函式？

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

---
# Part A — Mentor 示範
---

## A1. merge 的 validate 參數 — 確保 join 邏輯正確

merge 完筆數對不對？有沒有意外的重複？`validate` 可以幫你在 join 當下就檢查。

```python
pd.merge(
    left, right,
    on='key',
    how='left',
    validate='m:1'   # 左表多筆對右表一筆（最常見）
                     # 其他選項：'1:1'、'1:m'、'm:m'
)
# 如果 right 的 key 有重複，validate='m:1' 會直接 raise MergeError
# 比事後才發現 row 數爆增好得多
```

In [ ]:
telemetry = pd.read_csv('../data/raw/PdM_telemetry.csv', parse_dates=['datetime'])
machines  = pd.read_csv('../data/raw/PdM_machines.csv')

# 示範：telemetry left join machines，並驗證 machines 的 machineID 是唯一的
result = pd.merge(
    telemetry, machines,
    on='machineID',
    how='left',
    validate='m:1'   # 每台機器在 machines 裡只有一筆
)


print(f'telemetry: {telemetry.shape}')
print(f'after join: {result.shape}')   # row 數應該不變
print(f'新增欄位: {[c for c in result.columns if c not in telemetry.columns]}')

telemetry: (876100, 6)
after join: (876100, 8)
新增欄位: ['model', 'age']


In [4]:
telemetry.columns

Index(['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration'], dtype='object')

In [5]:
# 示範：如果 right 有重複 key，validate='m:1' 會報錯
machines_dup = pd.concat([machines, machines.head(3)])  # 故意製造重複

try:
    pd.merge(telemetry.head(10), machines_dup, on='machineID', how='left', validate='m:1')
except pd.errors.MergeError as e:
    print(f'MergeError 捕獲：{e}')

MergeError 捕獲：Merge keys are not unique in right dataset; not a many-to-one merge


## A2. join 後驗證筆數與缺值

join 完之後要做三件事確認結果正確：

```python
# 1. 筆數沒有爆增（left join 後 row 數應等於左表）
assert len(result) == len(left), f'row 數不符：{len(result)} vs {len(left)}'

# 2. 來自右表的欄位沒有意外缺值（代表有 key 沒對到）
result['right_col'].isnull().sum()

# 3. 檢查 join key 的範圍是否完整
left_keys  = set(left['key'].unique())
right_keys = set(right['key'].unique())
print('左有右無：', left_keys - right_keys)   # 這些 left row 會有 NaN
```

In [6]:
# 示範：join 後三步驗證
merged = pd.merge(telemetry, machines, on='machineID', how='left', validate='m:1')

# 1. 筆數驗證
assert len(merged) == len(telemetry), 'row 數爆增！'
print('✓ row 數正確')

# 2. 缺值驗證
null_counts = merged[['model', 'age']].isnull().sum()
print('右表欄位缺值：')
print(null_counts)

# 3. key 範圍
left_keys  = set(telemetry['machineID'].unique())
right_keys = set(machines['machineID'].unique())
print(f'\n左有右無的 machineID：{left_keys - right_keys}')

✓ row 數正確
右表欄位缺值：
model    0
age      0
dtype: int64

左有右無的 machineID：set()


## A3. pd.concat — 垂直合併多張同結構的表

當多張表的欄位結構相同，要疊在一起用 `concat`，不用 `merge`。

```python
pd.concat(
    [df1, df2, df3],
    ignore_index=True    # 重設 index（避免重複的 0,1,2...）
)
```

In [7]:
errors  = pd.read_csv('../data/raw/PdM_errors.csv',   parse_dates=['datetime'])
maint   = pd.read_csv('../data/raw/PdM_maint.csv',    parse_dates=['datetime'])
failures = pd.read_csv('../data/raw/PdM_failures.csv', parse_dates=['datetime'])

# 示範：errors 按照時間切成兩半，再合回來
first_half  = errors[errors['datetime'] < '2015-07-01']
second_half = errors[errors['datetime'] >= '2015-07-01']

print(f'前半：{len(first_half)} 筆，後半：{len(second_half)} 筆')

recombined = pd.concat([first_half, second_half], ignore_index=True)
print(f'合回來：{len(recombined)} 筆')
print(f'index 重設了嗎：{recombined.index.tolist()[:5]}')

前半：1953 筆，後半：1966 筆
合回來：3919 筆
index 重設了嗎：[0, 1, 2, 3, 4]


## A4. 用 source 欄位標記來源

concat 多張表時，有時需要知道每筆資料從哪張表來。

```python
df1['source'] = 'table_a'
df2['source'] = 'table_b'
combined = pd.concat([df1, df2], ignore_index=True)
```

In [10]:
# 示範：errors 和 failures 都有 datetime + machineID，結構類似
# 如果要合在一起分析「所有事件」，需要標記來源

errors_tagged   = errors.copy()
failures_tagged = failures.copy()

errors_tagged['event_type']   = 'error'
failures_tagged['event_type'] = 'failure'

# 注意：errors 有 errorID 欄，failures 有 failure 欄，欄位不完全一樣
# concat 時 pandas 會對齊欄位，缺的填 NaN
all_events = pd.concat([errors_tagged, failures_tagged], ignore_index=True)
print(all_events.shape)
print(all_events.isnull().sum())
print(all_events['event_type'].value_counts())
print(all_events['failure'][4675])

(4680, 5)
datetime         0
machineID        0
errorID        761
event_type       0
failure       3919
dtype: int64
event_type
error      3919
failure     761
Name: count, dtype: int64
comp3


## A5. 把操作包成函式 — 明確的輸入輸出契約

系統模組的核心是**函式介面**：輸入什麼、輸出什麼、出錯怎麼辦。

```python
def load_raw_data(data_dir: str) -> pd.DataFrame:
    """
    讀取所有原始表格並 join，回傳 raw DataFrame。

    Parameters
    ----------
    data_dir : str
        原始 CSV 所在的資料夾路徑

    Returns
    -------
    pd.DataFrame
        欄位：datetime, machineID, volt, rotate, pressure, vibration, model, age
        筆數：等於 telemetry 的筆數（876,100）
    """
    ...
```

**為什麼要寫 docstring？**  
不是給別人看的，是給三個月後的自己看的。
「回傳什麼欄位、筆數應該是多少」這些約定寫在 docstring 裡，
下游的 M3 才知道它能期待什麼輸入。

In [ ]:
# 示範：一個簡單但完整的 ingestion 函式
from pathlib import Path

def load_raw_data(data_dir: str) -> pd.DataFrame:
    """
    讀取原始表格並 join，回傳 raw DataFrame。

    Parameters
    ----------
    data_dir : str
        原始 CSV 所在的資料夾路徑

    Returns
    -------
    pd.DataFrame
        欄位：datetime, machineID, volt, rotate, pressure, vibration, model, age
        筆數：等於 telemetry 的筆數
    """
    p = Path(data_dir)

    telemetry = pd.read_csv(p / 'PdM_telemetry.csv', parse_dates=['datetime'])
    machines  = pd.read_csv(p / 'PdM_machines.csv')

    df = pd.merge(telemetry, machines, on='machineID', how='left', validate='m:1')

    assert len(df) == len(telemetry), 'join 後筆數不符'
    assert df[['model', 'age']].isnull().sum().sum() == 0, 'join 後出現缺值'

    return df


df = load_raw_data('../data/raw')
print(df.shape)
print(df.columns.tolist())
print(df.head(2))

---
# Part B — 你來練習
---

## B1. 驗證 maint join

把 `maint` left join 進 `telemetry`。

不對，等等 — 先想一下：maint 的 key 是 `machineID`，但 maint 每台機器有多筆，telemetry 也是每台機器多筆，這是什麼關係？能用 `validate='m:1'` 嗎？

試試看，觀察結果，印出 join 前後的 shape，解釋發生了什麼事。

In [18]:
# TODO
maint = pd.read_csv('../data/raw/PdM_maint.csv', parse_dates=['datetime'])
telemetry = pd.read_csv('../data/raw/PdM_telemetry.csv', parse_dates=['datetime'])

result = pd.merge(maint, telemetry, on=['machineID', 'datetime'], how='left', validate='m:1')

print(maint.shape, telemetry.shape)
result.shape

(3286, 3) (876100, 6)


(3286, 7)

## B2. pd.concat 練習

把 `errors`、`maint`、`failures` 三張表疊成一張 `all_events` DataFrame。
每筆資料要有 `source` 欄位標記來源（`'error'`、`'maint'`、`'failure'`）。

印出：
1. 合併後的 shape
2. 每個 source 各有幾筆
3. 每欄的缺值數量

In [23]:
# TODO
errors = pd.read_csv('../data/raw/PdM_errors.csv', parse_dates=['datetime'])
errors['source'] = 'error'
failures = pd.read_csv('../data/raw/PdM_failures.csv', parse_dates=['datetime'])
failures['source'] = 'failure'
maint = pd.read_csv('../data/raw/PdM_maint.csv', parse_dates=['datetime'])
maint['source'] = 'maint'



all_events = pd.concat([errors, maint, failures], ignore_index=True)

print(all_events.shape)
print(all_events.isnull().sum())

print(all_events.isnull().sum().sum())

print(all_events['source'].value_counts())



(7966, 6)
datetime        0
machineID       0
errorID      4047
source          0
comp         4680
failure      7205
dtype: int64
15932
source
error      3919
maint      3286
failure     761
Name: count, dtype: int64


## B3. 寫一個 load_telemetry_with_machines 函式

模仿 A5 的示範，寫一個函式：
- 參數：`data_dir: str`
- 回傳：只包含特定幾台機器的 telemetry + machines join 結果
  - 加一個 `machine_ids: list[int]` 參數，讓呼叫者指定要哪幾台機器
  - 預設值是 `None`（代表全部機器都要）
- 要有 docstring
- join 後要做驗證（筆數、缺值）

呼叫時用 `machine_ids=[1, 2, 3]` 測試，印出結果的 shape 和 machineID 分佈。

In [ ]:
# TODO
from pathlib import Path

def load_telemetry_with_machines(data_dir: str, machine_ids: list[int] = None) -> pd.DataFrame:
    """
    讀取原始表格並 join，回傳 DataFrame。

    Parameters
    ----------
    data_dir : str
        原始 CSV 所在的資料夾路徑

    machine_ids: list[int]
        想要的 machineID 清單，若為 None 則讀取全部機器

    Returns
    -------
    pd.DataFrame
        欄位：datetime, machineID, volt, rotate, pressure, vibration, model, age
        筆數：等於 telemetry 的筆數
    """
    p = Path(data_dir)

    telemetry = pd.read_csv(p / 'PdM_telemetry.csv', parse_dates=['datetime'])
    machines  = pd.read_csv(p / 'PdM_machines.csv')

    if machine_ids is not None:
        telemetry = telemetry[telemetry['machineID'].isin(machine_ids)]

    df = pd.merge(telemetry, machines, on='machineID', how='left', validate='m:1')

    assert len(df) == len(telemetry), 'join 後筆數不符'
    assert df[['model', 'age']].isnull().sum().sum() == 0, 'join 後出現缺值'

    return df

a = [1, 2, 3]

df = load_telemetry_with_machines('../data/raw', machine_ids=a)
print(df.shape)
print(df.columns.tolist())
print(df.head(2))
print(df['machineID'].unique())
print(df['machineID'].value_counts())

(26283, 8)
['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration', 'model', 'age']
             datetime  machineID    volt  rotate  pressure  vibration   model  \
0 2015-01-01 06:00:00          1 176.218 418.504   113.078     45.088  model3   
1 2015-01-01 07:00:00          1 162.879 402.747    95.461     43.414  model3   

   age  
0   18  
1   18  
[1 2 3]
machineID
1    8761
2    8761
3    8761
Name: count, dtype: int64
